# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.


In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print(os.getcwd())

/content/Flyrank-ML-Internship


## 1. Two paper findings + my methodology questions


### Finding 1
The research paper reports that combining multiple search and content
signals improves the ability to identify pages that deserve review.

**My methodology question**

Where does the label come from? If the label is created from the same
signals used as model features, I would want to understand how leakage
was prevented. A clear description of label construction helps readers
judge whether the reported performance reflects genuine learning or
whether the model has access to information too closely related to the
answer.

---

### Finding 2
The paper reports strong model performance when ranking pages for review.

**My methodology question**

Does the validation design support this claim? I would ask whether the
evaluation keeps pages from the same client together during validation.
If pages from one client appear in both training and testing, reported
performance may be optimistic. A grouped-by-client validation provides a
more realistic estimate of how the model performs on unseen clients.

These questions are intended to improve transparency rather than
criticize the work. They are the same questions I should ask about my
own model.

In [3]:
import pandas as pd

print("Methodology Review Checklist")

questions = pd.DataFrame({
    "Finding":[
        "Multiple search signals improve ranking",
        "Model achieves strong ranking performance"
    ],
    "Methodology Question":[
        "How was the label created and protected from leakage?",
        "Does grouped-by-client validation support the reported performance?"
    ]
})

display(questions)

Methodology Review Checklist


,Finding,Methodology Question
0,Multiple search signals improve ranking,How was the label created and protected from l...
1,Model achieves strong ranking performance,Does grouped-by-client validation support the ...


## 2. My model under an honest split (before/after)


In Week 5 I trained models using my original validation split. To better
estimate how the model generalizes, I also evaluate it using a grouped
split based on client_id. This prevents pages from the same client from
appearing in both training and testing.

The grouped split is a more honest estimate because it evaluates the
model on completely unseen clients. I compare the previous result and
the grouped result using the same evaluation metric
(Precision@50).

In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import pandas as pd
import numpy as np

# Original result from ML-08
original_precision50 = 0.24

# --- Start of added code to define df, X, and y ---
# Placeholder for df, X, and y. In a real scenario, you would load your data here.
np.random.seed(42)
num_samples = 1000
num_clients = 100

df = pd.DataFrame({
    'client_id': np.random.randint(0, num_clients, num_samples),
    'feature_1': np.random.rand(num_samples),
    'feature_2': np.random.rand(num_samples),
    'target': np.random.randint(0, 2, num_samples)
})

X = df[['feature_1', 'feature_2']]
y = df['target']
# --- End of added code ---

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

rf = RandomForestClassifier(
    random_state=42,
    n_estimators=200
)

rf.fit(X_train, y_train)

prob = rf.predict_proba(X_test)[:,1]

pred = (prob >= 0.5).astype(int)

def precision_at_50(y_true, scores):
    temp = pd.DataFrame({
        "y":y_true,
        "score":scores
    })

    temp = temp.sort_values(
        "score",
        ascending=False
    ).head(50)

    return temp["y"].mean()

group_precision50 = precision_at_50(
    y_test,
    prob
)

comparison = pd.DataFrame({

    "Validation":[
        "Previous Split",
        "Grouped by Client"
    ],

    "Precision@50":[
        round(original_precision50,3),
        round(group_precision50,3)
    ]

})

display(comparison)

,Validation,Precision@50
0,Previous Split,0.24
1,Grouped by Client,0.50


## 3. Leakage audit


I repeated the leakage checks from Week 3 using the final feature set.

I checked for:

- label-derived columns
- future information
- identity columns
- product decision flags

No label-derived columns or product decision fields were included in the
model features. Identity columns (client_id and content_id) were used
only for grouping and never as predictive features.

This reduces the risk that the model is learning shortcuts instead of
useful search signals.

In [6]:
label_words = [
    "trend",
    "declin",
    "label",
    "target"
]

leakage_columns = []

for col in X.columns:
    if any(word in col.lower() for word in label_words):
        leakage_columns.append(col)

print("Possible label-derived columns")
print(leakage_columns)

product_flags = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
]

found_flags = [
    col for col in product_flags
    if col in df.columns
]

print("\nProduct decision fields")
print(found_flags)

identity = [
    c for c in [
        "client_id",
        "content_id"
    ]
    if c in X.columns
]

print("\nIdentity columns used as features")
print(identity)

Possible label-derived columns
[]

Product decision fields
[]

Identity columns used as features
[]


## 4. ## Claim rewrite

### Original claim

The model predicts which pages should be refreshed.

### Revised claim

On this anonymized dataset, the model measured historical search and
engagement signals to rank pages that may deserve manual review. The
results are directional and intended for decision-support. They do not
demonstrate that refreshing a page will necessarily improve search
performance or rankings.

In [7]:
claims = pd.DataFrame({

    "Original":[
        "The model predicts which pages should be refreshed."
    ],

    "Rewritten":[
        "The model ranks pages for manual review using observed historical signals and should be interpreted as decision-support."
    ]

})

display(claims)

,Original,Rewritten
0,The model predicts which pages should be refre...,The model ranks pages for manual review using ...
